In [2]:
import pandas as pd
import numpy as np
import glob
import os
import gc

# ============================================================
# CONFIGURACIÓN
# ============================================================

ruta = "D:/Licit/*.csv"

columns_to_drop = ['Codigo', 'Link', 'Nombre', 'Descripcion', 'Tipo de Adquisicion', 'CodigoEstado', 'Estado', 'NombreOrganismo', 'RutUnidad', 'NombreUnidad', 'DireccionUnidad', 'RegionUnidad', 'Informada', 'CodigoTipo', 'TipoConvocatoria', 'CodigoMoneda', 'Etapas', 'EstadoEtapas', 'TomaRazon', 'EstadoPublicidadOfertas', 'JustificacionPublicidad', 'EstadoCS', 'Contrato', 'Obras', 'FechaCreacion', 'FechaInicio', 'FechaFinal', 'FechaPubRespuestas', 'FechaActoAperturaTecnica', 'FechaActoAperturaEconomica', 'FechaAdjudicacion', 'FechaEstimadaAdjudicacion', 'FechaSoporteFisico', 'FechaTiempoEvaluacion', 'UnidadTiempoEvaluacion', 'FechaEstimadaFirma', 'FechasUsuario', 'FechaVisitaTerreno', 'DireccionVisita', 'FechaEntregaAntecedentes', 'DireccionEntrega', 'Estimacion', 'FuenteFinanciamiento', 'VisibilidadMonto', 'Tiempo', 'UnidadTiempo', 'Modalidad', 'TipoPago', 'ProhibicionContratacion', 'SubContratacion', 'UnidadTiempoDuracionContrato', 'TiempoDuracionContrato', 'TipoDuracionContrato', 'JustificacionMontoEstimado', 'ObservacionContrato', 'ExtensionPlazo', 'EsBaseTipo', 'UnidadTiempoContratoLicitacion', 'ValorTiempoRenovacion', 'PeriodoTiempoRenovacion', 'EsRenovable', 'TipoAprobacion', 'NumeroAprobacion', 'FechaAprobacion', 'CodigoProveedor', 'Correlativo', 'CodigoEstadoLicitacion', 'Codigoitem', 'Rubro1', 'Rubro2', 'Rubro3', 'Nombre producto genrico', 'Nombre linea Adquisicion', 'Descripcion linea Adquisicion', 'UnidadMedida', 'CodigoSucursalProveedor', 'RutProveedor', 'NombreProveedor', 'RazonSocialProveedor', 'DescripcionProveedor', 'Monto Estimado Adjudicado', 'Nombre de la Oferta', 'Estado Oferta', 'Moneda de la Oferta', 'MontoUnitarioOferta', 'CantidadAdjudicada', 'MontoLineaAdjudica', 'FechaEnvioOferta']

# ============================================================
# FUNCIÓN ADAPTADA PARA ESTE DATASET
# Inspirada en la estrategia de "Funciones Tipos.py" del cliente:
#   1) convertir fechas
#   2) reducir enteros cuando sea seguro
#   3) usar category en texto de baja cardinalidad
#   4) conservar float64 en montos/cantidades para no perder precisión
#
# NO usa los nombres OC_/Item_/Proveedor_ del archivo del cliente.
# Trabaja únicamente con las columnas que realmente existan en este dataset.
# ============================================================



def optimizar_tipos_licitaciones(df):
    df = df.copy()

    memoria_antes = df.memory_usage(deep=True).sum() / 1024**2

    # --------------------------------------------------------
    # 1. FECHAS QUE HAYAMOS CONSERVADO
    # --------------------------------------------------------
    columnas_fecha = [
        "FechaPublicacion",
        "FechaCierre",
    ]

    for col in columnas_fecha:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # --------------------------------------------------------
    # 2. CÓDIGOS NUMÉRICOS
    # downcast solo si pandas puede hacerlo de forma segura.
    # --------------------------------------------------------
    columnas_codigos = [
        "CodigoOrganismo",
        "CodigoUnidad",
        "CodigoProductoONU",
    ]

    for col in columnas_codigos:
        if col in df.columns:
            convertido = pd.to_numeric(df[col], errors="coerce")
            if convertido.notna().all():
                df[col] = pd.to_numeric(convertido, downcast="unsigned")
            else:
                # UInt32/UInt64 permiten NA, a diferencia de uint32/uint64 de NumPy.
                maximo = convertido.max()
                if pd.notna(maximo) and maximo <= np.iinfo(np.uint32).max and (convertido.dropna() >= 0).all():
                    df[col] = convertido.astype("UInt32")
                else:
                    df[col] = convertido.astype("Int64")

    # --------------------------------------------------------
    # 3. TEXTO DE BAJA CARDINALIDAD -> CATEGORY
    # Criterio conservador: máximo 5% de valores únicos
    # y al menos 2 valores distintos.
    # --------------------------------------------------------
    for col in df.select_dtypes(include=["object", "string"]).columns:
        nunique = df[col].nunique(dropna=True)
        if len(df) > 0 and 1 < nunique <= max(100, int(len(df) * 0.05)):
            df[col] = df[col].astype("category")

    # --------------------------------------------------------
    # 4. VARIABLES NUMÉRICAS DE MEDICIÓN/MONTO
    # Se mantienen float64: aquí priorizamos precisión.
    # --------------------------------------------------------
    columnas_float64 = [
        "MontoEstimado",
        "Cantidad",
        "Cantidad Ofertada",
        "Valor Total Ofertado",
        "NumeroOferentes",
        "CantidadReclamos",
    ]

    for col in columnas_float64:
        if col in df.columns:

            serie = (
                df[col]
                .astype("string")
                .str.strip()
                .str.replace(",", ".", regex=False)
            )

            df[col] = pd.to_numeric(
                serie,
                errors="coerce"
            ).astype("float64")

    memoria_despues = df.memory_usage(deep=True).sum() / 1024**2

    print(
        f"Memoria: {memoria_antes:,.2f} MB -> "
        f"{memoria_despues:,.2f} MB"
    )

    return df


# ============================================================
# ARCHIVOS
# ============================================================

archivos = sorted(glob.glob(ruta))

print(f"Archivos encontrados: {len(archivos)}")

if not archivos:
    raise FileNotFoundError("No se encontraron archivos CSV.")

dataframes = []

for archivo in archivos:
    nombre = os.path.basename(archivo)

    print("\n" + "=" * 80)
    print(f"Leyendo: {nombre}")
    print("=" * 80)

    df_mes = pd.read_csv(
        archivo,
        sep=";",
        encoding="cp1252",
        encoding_errors="replace",
        low_memory=False
    )

    print(f"Filas originales: {len(df_mes):,}")
    print(f"Columnas originales: {len(df_mes.columns):,}")

    # --------------------------------------------------------
    # CARACTERES DE CODIFICACIÓN PROBLEMÁTICOS
    # --------------------------------------------------------
    columnas_texto = df_mes.select_dtypes(include=["object", "string"]).columns

    for columna in columnas_texto:
        mascara = (
            df_mes[columna]
            .astype("string")
            .str.contains("�", regex=False, na=False)
        )

        cantidad = int(mascara.sum())

        if cantidad:
            print(f"  ⚠ {columna}: {cantidad:,} valores corruptos -> NaN")
            df_mes.loc[mascara, columna] = np.nan

    # --------------------------------------------------------
    # FILAS SIN OFERTA
    # Debe hacerse antes de eliminar columnas.
    # --------------------------------------------------------
    requeridas_oferta = ["Cantidad Ofertada", "Valor Total Ofertado"]
    faltantes = [c for c in requeridas_oferta if c not in df_mes.columns]

    if faltantes:
        raise KeyError(
            f"En {nombre} faltan columnas necesarias: {faltantes}"
        )

    mask_sin_oferta = (
        df_mes["Cantidad Ofertada"].isna()
        & df_mes["Valor Total Ofertado"].isna()
    )

    print(f"Filas sin oferta eliminadas: {mask_sin_oferta.sum():,}")
    df_mes = df_mes.loc[~mask_sin_oferta].copy()

    # --------------------------------------------------------
    # ELIMINAR COLUMNAS INNECESARIAS ANTES DEL CONCAT
    # --------------------------------------------------------
    presentes = [c for c in columns_to_drop if c in df_mes.columns]
    df_mes.drop(columns=presentes, inplace=True)

    print(f"Columnas eliminadas: {len(presentes):,}")
    print(f"Columnas restantes: {len(df_mes.columns):,}")

    # --------------------------------------------------------
    # OPTIMIZACIÓN MES A MES
    # También reduce la memoria antes de guardar el mes.
    # --------------------------------------------------------

    # ============================================================
    # DIAGNÓSTICO MONTO ESTIMADO ANTES DE OPTIMIZAR TIPOS
    # ============================================================

    original = df_mes["MontoEstimado"]

    nulos_originales = original.isna().sum()

    # --------------------------------------------------------
    # DIAGNÓSTICO ANTES DE OPTIMIZAR
    # --------------------------------------------------------

    serie = df_mes["MontoEstimado"]

    convertido = pd.to_numeric(
        serie,
        errors="coerce"
    )

    invalidos_conversion = (
        serie.notna()
        & convertido.isna()
    )

    print("\nMontoEstimado:")
    print("NaN originales:", serie.isna().sum())
    print(
        "Valores que fallan al convertir:",
        invalidos_conversion.sum()
    )

    print(
        serie.loc[invalidos_conversion]
        .value_counts(dropna=False)
        .head(20)
    )

    # --------------------------------------------------------
    # AHORA SÍ OPTIMIZAR
    # --------------------------------------------------------

    df_mes = optimizar_tipos_licitaciones(df_mes)

    normalizado = (
        original
        .astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False)
    )

    convertido_directo = pd.to_numeric(
        original,
        errors="coerce"
    )

    convertido_normalizado = pd.to_numeric(
        normalizado,
        errors="coerce"
    )

    print("\nDIAGNÓSTICO MontoEstimado")
    print("NaN originales:", nulos_originales)

    print(
        "NaN conversión DIRECTA:",
        convertido_directo.isna().sum()
    )

    print(
        "NaN conversión NORMALIZADA:",
        convertido_normalizado.isna().sum()
    )

    print(
        "Valores recuperados al normalizar:",
        (
            convertido_directo.isna()
            & convertido_normalizado.notna()
        ).sum()
    )

    df_mes = optimizar_tipos_licitaciones(df_mes)

    dataframes.append(df_mes)

    gc.collect()

# ============================================================
# VALIDAR ESQUEMA ANTES DE CONCATENAR
# ============================================================

columnas_base = list(dataframes[0].columns)

for i, df_mes in enumerate(dataframes[1:], start=2):
    faltan = sorted(set(columnas_base) - set(df_mes.columns))
    sobran = sorted(set(df_mes.columns) - set(columnas_base))

    if faltan or sobran:
        raise ValueError(
            f"El DataFrame mensual #{i} tiene un esquema diferente. "
            f"Faltan={faltan}; Sobran={sobran}"
        )

# ============================================================
# CONCATENAR
# ============================================================

df = pd.concat(dataframes, ignore_index=True)
del dataframes
gc.collect()

# Reaplicar después del concat para unificar categorías/tipos.
df = optimizar_tipos_licitaciones(df)

# ============================================================
# RESUMEN
# ============================================================

print("\n" + "=" * 80)
print("DATASET ANUAL DEPURADO")
print("=" * 80)

print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns):,}")
print(f"Dimensiones: {df.shape}")

print("\nTipos finales:")
print(df.dtypes)

print("\nNaN variables de oferta:")
print(df[["Cantidad Ofertada", "Valor Total Ofertado"]].isna().sum())

# ============================================================
# GUARDAR
# ============================================================

salida = "licitaciones_anual_depurado.feather"
df.to_feather(salida)

print(f"\n✓ Guardado como {salida}")


Archivos encontrados: 12

Leyendo: lic_2024-01.csv
Filas originales: 203,486
Columnas originales: 105
  ⚠ Descripcion linea Adquisicion: 9 valores corruptos -> NaN
  ⚠ DescripcionProveedor: 11 valores corruptos -> NaN
Filas sin oferta eliminadas: 0
Columnas eliminadas: 88
Columnas restantes: 17

MontoEstimado:
NaN originales: 114
Valores que fallan al convertir: 39092
MontoEstimado
1,5e+07     2653
2,5e+07     2612
6,3e+07     1979
3,2e+07     1928
1,2e+07     1365
4,2e+07     1333
1,4e+07     1260
2,1e+08     1107
6,4e+07      961
1,8e+07      940
1,34e+08     929
1,2e+08      875
1,1e+07      859
1,3e+07      724
5,5e+07      679
1,1e+08      659
4,5e+07      645
1,6e+07      630
3,1e+07      586
3,4e+07      573
Name: count, dtype: int64
Memoria: 45.55 MB -> 19.80 MB

DIAGNÓSTICO MontoEstimado
NaN originales: 114
NaN conversión DIRECTA: 39206
NaN conversión NORMALIZADA: 114
Valores recuperados al normalizar: 39092
Memoria: 19.80 MB -> 19.80 MB

Leyendo: lic_2024-02.csv
Filas origina